# 001a. Update the ADS record snapshot

Yearly update notebook mirroring `001-collect-ads-records.ipynb`.


## Purpose, inputs, and outputs

**Purpose**
- Check the coverage of the canonical all-years snapshot.
- Fetch one new ADS year-slice.
- Merge, hydrate, and fetch citation histories using the same logic as the main notebook.

**Requires**
- The canonical hydrated JSON extracted from the retained stage-001 snapshot archive.
- An ADS API token in `ADS_TOKEN`, or pasted into `USER_ADS_TOKEN` in the setup cell.

**Writes**
- `code/stage-outputs/001a-update-ads-records/data_<year>.json`
- `code/stage-outputs/001a-update-ads-records/merged_records_with_abstracts.json`
- `code/stage-outputs/001a-update-ads-records/metrics_citations.json`


In [ ]:
import os

from shared.ads_api import (
    DEFAULT_METRICS_TYPES,
    DEFAULT_SEARCH_FIELDS,
    build_year_query,
    fetch_metrics_for_bibcodes,
    fetch_missing_abstracts,
    latest_record_year,
    load_json,
    merge_unique_records,
    query_ads_api,
    save_json,
    set_ads_token,
    summarize_record_years,
)
from shared.project_paths import get_project_paths

paths = get_project_paths()
output_dir = paths.output_dir("001a-update-ads-records")
canonical_output_dir = paths.output_dir("001-collect-ads-records")

USER_ADS_TOKEN = ""  # Optional: paste your ADS token here for this session only.
if USER_ADS_TOKEN.strip():
    set_ads_token(USER_ADS_TOKEN)

ADS_TOKEN = os.environ.get("ADS_TOKEN", "")
APPEND_YEAR = 2025
SEARCH_FIELDS = DEFAULT_SEARCH_FIELDS
METRIC_TYPES = DEFAULT_METRICS_TYPES

CANONICAL_HYDRATED_JSON = canonical_output_dir / "ads_search_all_years_with_abstracts.json"
LEGACY_JSON = CANONICAL_HYDRATED_JSON
NEW_YEAR_JSON = output_dir / f"ads_search_{APPEND_YEAR}.json"
MERGED_JSON = output_dir / "merged_records.json"
HYDRATED_JSON = output_dir / "merged_records_with_abstracts.json"
METRICS_JSON = output_dir / "metrics_citations.json"

print("Canonical hydrated JSON:", CANONICAL_HYDRATED_JSON)
print("New year JSON:", NEW_YEAR_JSON)
print("Merged JSON:", MERGED_JSON)
print("Hydrated JSON:", HYDRATED_JSON)
print("Metrics JSON:", METRICS_JSON)
print("Using ADS token from environment:", bool(ADS_TOKEN))
print("Search fields:", SEARCH_FIELDS)
print("Metrics types:", METRIC_TYPES)


## Stage 1. Check the canonical snapshot before appending


In [ ]:
legacy_records = load_json(LEGACY_JSON, default=[])
year_summary = summarize_record_years(legacy_records)
latest_year = latest_record_year(legacy_records)

if year_summary is None:
    print("Canonical snapshot not found or contains no usable year values.")
else:
    print(
        "Canonical snapshot covers years "
        f"{year_summary['year_min']} to {year_summary['year_max']} "
        f"across {year_summary['n_years']} distinct years."
    )
    if APPEND_YEAR <= latest_year:
        print(
            f"Warning: APPEND_YEAR={APPEND_YEAR} is already present in the canonical snapshot. "
            "You may want to stop here to avoid an unnecessary ADS request."
        )
    else:
        print(f"Append target year {APPEND_YEAR} is newer than the canonical snapshot.")


## Stage 2. Fetch the new year-slice


In [ ]:
if not ADS_TOKEN:
    raise ValueError(
        "Set ADS_TOKEN in your environment, or paste it into USER_ADS_TOKEN in the setup cell."
    )

encoded_query = build_year_query(APPEND_YEAR, fields=SEARCH_FIELDS)
new_records = query_ads_api(encoded_query, ADS_TOKEN)
save_json(new_records, NEW_YEAR_JSON)

print(f"Saved {len(new_records):,} new ADS records to {NEW_YEAR_JSON}")


## Stage 3. Merge with the canonical snapshot


In [ ]:
legacy_records = load_json(LEGACY_JSON, default=[])
new_records = load_json(NEW_YEAR_JSON, default=[])
merged_records = merge_unique_records(legacy_records, new_records)
save_json(merged_records, MERGED_JSON)

print(f"Legacy records: {len(legacy_records):,}")
print(f"New records: {len(new_records):,}")
print(f"Merged unique records: {len(merged_records):,}")


## Stage 4. Hydrate abstracts and fetch citation histories


In [ ]:
merged_records = load_json(MERGED_JSON, default=[])
hydrated_records = fetch_missing_abstracts(merged_records, ADS_TOKEN)
save_json(hydrated_records, HYDRATED_JSON)

missing_after = sum(1 for record in hydrated_records if not record.get("abstract"))
print(f"Saved hydrated merged records to {HYDRATED_JSON}")
print(f"Records still missing abstracts: {missing_after:,}")


In [ ]:
hydrated_records = load_json(HYDRATED_JSON, default=[])
bibcodes = [record.get("bibcode") for record in hydrated_records if record.get("bibcode")]
metrics_citations = fetch_metrics_for_bibcodes(
    bibcodes,
    ADS_TOKEN,
    metric_types=METRIC_TYPES,
)
save_json(metrics_citations, METRICS_JSON)

print(f"Saved citation metrics payload for {len(bibcodes):,} bibcodes to {METRICS_JSON}")


## Stage 5. Inspect the staged append outputs


In [ ]:
hydrated_records = load_json(HYDRATED_JSON, default=[])
print(f"Hydrated record count: {len(hydrated_records):,}")
for i, record in enumerate(hydrated_records[:3], 1):
    print(f"\nRecord {i}")
    print("bibcode:", record.get("bibcode"))
    print("year:", record.get("year"))
    abstract = record.get("abstract") or ""
    print("abstract preview:", abstract[:240])
